In [ ]:
%load_ext autoreload
%autoreload 2

# The xBD-S12 dataset

In [ ]:
import matplotlib.pyplot as plt

from src.constants import CLASSES_ORIGINAL, COLORS_ORIGINAL
from src.data.metadata import load_metadata
from src.training.dataloaders import get_dataloaders
from src.training.utils import unbind_samples

## Visualization

## Examples

In [ ]:
# Example usage
batch_size = 4
dataloaders = get_dataloaders(
    modalities=["s1", "s2_tci", "xbd"],
    batch_size=batch_size,
    downsample_xbd = False,
    remove_tiles_without_buildings=True,
    pixels_buffer_around_buildings=3,
)
dl = dataloaders["train"]

for _batch in dl:
    break

samples = unbind_samples(_batch)
n_imgs = dl.dataset.get_n_imgs()
fig, axs = plt.subplots(batch_size, n_imgs, figsize=(3 * n_imgs, 12))
for i, sample in enumerate(samples):
    dl.dataset.plot(sample, axs=axs[i])
plt.tight_layout()
plt.show()

## Metadata
The dataset is based on a single `.geojson` file containing the metadata for all the tiles.

In [ ]:
gdf = load_metadata()
print(gdf.shape)
gdf.head()

### Number of image pairs
Each image pair (pre- and post-disaster) is referred by a unique ID (`xbd_uid`). The dataset contains 10,315 pairs.

In [ ]:
total_image_pairs_per_disaster = gdf.groupby("disaster").size()
fig, ax = plt.subplots(figsize=(10, 6))
total_image_pairs_per_disaster.plot(kind="bar", ax=ax, color="skyblue")
ax.set_title(f"Number of image pairs per Disaster (Total={total_image_pairs_per_disaster.sum()})", fontweight='bold')
ax.set_xlabel("Disaster")
ax.set_ylabel("Number of Image Pairs")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Building Instances
We keep track of how many building instances there are per tile and per damage class.

#### Number of building instances per damage class and per disaster

In [ ]:
columns_count = ['N_intact', 'N_minor', 'N_major', 'N_destroyed', 'N_unclassified']
total_per_class = gdf[columns_count].sum()
damage_per_disaster = gdf.groupby('disaster')[columns_count].sum()  # shape: (n_disasters, 5)

labels = ['Intact', 'Minor Damage', 'Major Damage', 'Destroyed', 'Unclassified']
classes_to_int = {name: idx for idx, name in CLASSES_ORIGINAL.items()}
colors = [COLORS_ORIGINAL[classes_to_int[label]] for label in labels]


# Create the figure with two side-by-side subplots
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Subplot 1: Global distribution (Pie Chart) (text in white)
axes[0].pie(
    total_per_class,
    labels=None,
    colors=colors,
    startangle=140,
    autopct='%1.1f%%',
    pctdistance=0.85,
    textprops={'color': 'white', 'fontsize': 12}
)
axes[0].set_title('Overall Building Damage Distribution', fontsize=14, fontweight='bold')
axes[0].legend(labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.15))

# Subplot 2: Total building count per disaster (Bar Chart)
damage_per_disaster.plot(
    kind='bar',
    stacked=True,
    ax=axes[1],
    color=colors,
    legend=False,  # we'll add a shared legend manually
)
axes[1].set_title('Total Number of Buildings per Disaster', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Number of Buildings')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

#### Number of building instances per damage class per disaster
Same as above but pie charts

In [ ]:
columns_count = ['N_intact', 'N_minor', 'N_major', 'N_destroyed', 'N_unclassified']
total_per_class_per_disaster = gdf.groupby('disaster')[columns_count].sum()

labels = ['Intact', 'Minor Damage', 'Major Damage', 'Destroyed', 'Unclassified']
classes_to_int = {name: idx for idx, name in CLASSES_ORIGINAL.items()}
colors = [COLORS_ORIGINAL[classes_to_int[label]] for label in labels]

# Create a 4x4 grid of subplots
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
axes = axes.flatten()

# Plot each disaster as a pie chart
for i, (_, row) in enumerate(total_per_class_per_disaster.iterrows()):
    if i < 16:
        axes[i].pie(row[columns_count], colors=colors, startangle=140)
        axes[i].set_title(f"{row.name}\n(N={row.sum()})", fontsize=12, fontweight='bold')
    else:
        axes[i].axis('off')

# Add a common legend for the entire figure
fig.legend(labels, loc='center', ncol=5, fontsize=14, bbox_to_anchor=(0.5, 0))
plt.tight_layout()
plt.show()